SalesInsight PY
Sales Data Analysis and Visualization Pipeline

Assessment Mini-Project - Module 01 - SC Tech

This file implements a complete sales data analysis pipeline, covering:

- Synthetic dataset generation with intentionally dirty data
- CSV loading and initial inspection
- Data cleaning with datetime and regex
- Derived column creation
- Aggregated metrics with groupby
- Customer segmentation with lambda functions
- NumPy statistics, vectorized operations, and broadcasting
- Visualizations with Matplotlib and Seaborn
- Object-Oriented Programming
- Inheritance with super()
- Higher-order function / callback
- CSV and JSON exports
- Full execution through main()

Renan de Brito Leme

In [1]:
# ============================================================
# RF00 - Libraries import
# ============================================================

import os
import re
import json
import random
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
# ============================================================
# RF01 - Create or load the sales dataset
# ============================================================
def generate_sales_dataset(n_records=200, seed=42):
    """
    Generates a synthetic sales dataset with intentionally dirty data.
    """
    random.seed(seed)
    np.random.seed(seed)

    products = [
        "Notebook", "Smartphone", "Tablet", "Monitor",
        "Keyboard", "Mouse", "Headset"
    ]

    categories = {
        "Notebook": "Computers",
        "Smartphone": "Mobile Devices",
        "Tablet": "Mobile Devices",
        "Monitor": "Computers",
        "Keyboard": "Peripherals",
        "Mouse": "Peripherals",
        "Headset": "Peripherals"
    }

    base_prices = {
        "Notebook": 3500,
        "Smartphone": 2200,
        "Tablet": 1800,
        "Monitor": 1200,
        "Keyboard": 250,
        "Mouse": 120,
        "Headset": 350
    }

    regions = ["Southeast", "South", "Northeast", "Midwest", "North"]
    customers = [f"Customer_{i:03d}" for i in range(1, 51)]
    start_date = datetime(2024, 1, 1)

    data = []

    for i in range(n_records):
        product = random.choice(products)
        quantity = random.randint(1, 10)
        unit_price = round(base_prices[product] * random.uniform(0.85, 1.15), 2)
        sale_date = start_date + timedelta(days=random.randint(0, 364))

        # Insert intentionally dirty data
        if random.random() < 0.05:
            quantity = None

        if random.random() < 0.04:
            unit_price = None

        if random.random() < 0.03:
            product = "  " + product

        # Important: this line must be outside the if blocks
        sale_date_value = (
            sale_date.strftime("%Y-%m-%d")
            if random.random() > 0.02
            else "INVALID DATE"
        )

        data.append({
            "sale_id": i + 1,
            "sale_date": sale_date_value,
            "customer": random.choice(customers),
            "product": product,
            "category": categories.get(product.strip(), "Other"),
            "region": random.choice(regions),
            "quantity": quantity,
            "unit_price": unit_price
        })

    return pd.DataFrame(data)

def generate_and_save_dataset(file_path="sales.csv", n_records=200):
    """
    Generates the synthetic dataset and saves it as a CSV file.
    """
    raw_df = generate_sales_dataset(n_records=n_records)
    raw_df.to_csv(file_path, index=False, encoding="utf-8-sig")

    print("\n=== DATASET GENERATED ===")
    print(f"File saved at: {file_path}")
    print(f"Dataset generated with {len(raw_df)} records.")
    print(raw_df.head())

    return raw_df

In [3]:
# ============================================================
# RF02 - Inspect and describe the data
# ============================================================
def inspect_data(df):
    """
    Displays basic information about the DataFrame.
    """
    print("\n=== INITIAL DATASET INSPECTION ===")
    print(f"Shape: {df.shape}")
    print(f"\nColumns: {list(df.columns)}")
    print(f"\nData types:\n{df.dtypes}")
    print(f"\nMissing values by column:\n{df.isnull().sum()}")
    print(f"\nFirst records:\n{df.head()}")
    print(f"\nDescriptive statistics:\n{df.describe(include='all')}")

In [5]:
# ============================================================
# RF03 - Clean and process the data
# ============================================================
def clean_data(df):
    """
    Cleans and processes the sales DataFrame.
    Operations performed:
    - removes extra spaces from text columns;
    - converts sale_date to datetime;
    - removes invalid dates;
    - removes rows with missing quantity or unit_price;
    - converts numeric columns to proper types.
    Returns:
        tuple: Clean DataFrame and cleaning report.
    """
    initial_count = len(df)
    cleaning_report = {}

    # 1. Remove extra spaces from text columns
    text_columns = df.select_dtypes(include="object").columns
    for column in text_columns:
        df[column] = df[column].astype(str).str.strip()
        
    # 2. Convert date and remove invalid dates
    df["sale_date"] = pd.to_datetime(df["sale_date"], errors="coerce")
    invalid_dates_count = df["sale_date"].isnull().sum()
    df = df.dropna(subset=["sale_date"])
    cleaning_report["invalid_dates_removed"] = int(invalid_dates_count)

    # 3. Remove rows with missing quantity or unit price
    before_null_removal = len(df)
    df = df.dropna(subset=["quantity", "unit_price"])
    cleaning_report["rows_with_nulls_removed"] = int(before_null_removal - len(df))

    # 4. Ensure correct numeric types
    df["quantity"] = df["quantity"].astype(int)
    df["unit_price"] = df["unit_price"].astype(float)
    final_count = len(df)
    cleaning_report["initial_records"] = int(initial_count)
    cleaning_report["final_records"] = int(final_count)
    cleaning_report["total_records_removed"] = int(initial_count - final_count)

    # 5. Cleaning report
    print("\n=== CLEANING REPORT ===")
    for key, value in cleaning_report.items():
        print(f"  {key}: {value}")
        
    return df, cleaning_report


In [7]:
# ============================================================
# RF04 - Create derived columns with transformations
# ============================================================

def create_derived_columns(df):
    """
    Creates calculated and derived columns from the cleaned dataset.
    """
    # Total revenue per sale row
    df["total_revenue"] = df["quantity"] * df["unit_price"]
    # Extract date components using datetime
    df["month"] = df["sale_date"].dt.month
    df["month_name"] = df["sale_date"].dt.strftime("%B")
    df["quarter"] = df["sale_date"].dt.quarter.apply(lambda q: f"Q{q}")
    df["year"] = df["sale_date"].dt.year
    # Vectorized conditional transformation with np.select
    conditions = [
        df["total_revenue"] < 500,
        (df["total_revenue"] >= 500) & (df["total_revenue"] < 5000),
        df["total_revenue"] >= 5000,
    ]
    labels = ["Low Value", "Medium Value", "High Value"]
    df["item_revenue_range"] = np.select(
        conditions,
        labels,
        default="Unclassified",
    )
    print("\n=== DERIVED COLUMNS CREATED ===")
    print(
        df[
            [
                "sale_date",
                "total_revenue",
                "month",
                "month_name",
                "quarter",
                "year",
                "item_revenue_range",
            ]
        ].head()
    )
    return df

In [9]:
# ============================================================
# RF05 - Calculate aggregated metrics with groupby
# ============================================================

def calculate_metrics(df):
    """
    Calculates and returns aggregated metrics from the dataset.
    """
    
    # Revenue by month
    metrics = {}

    by_month = (
        df.groupby("month")
        .agg(
            total_revenue=("total_revenue", "sum"),
            quantity=("quantity", "sum"),
            number_of_sales=("sale_id", "count"),
        )
        .reset_index()
        .sort_values("month")
    )
    metrics["by_month"] = by_month

    # Revenue by quarter
    by_quarter = (
        df.groupby("quarter")
        .agg(
            total_revenue=("total_revenue", "sum"),
            quantity=("quantity", "sum"),
            number_of_sales=("sale_id", "count"),
        )
        .reset_index()
        .sort_values("quarter")
    )
    metrics["by_quarter"] = by_quarter

    # Top 5 products by revenue
    top_products = (
        df.groupby("product")["total_revenue"]
        .sum()
        .sort_values(ascending=False)
        .head(5)
        .reset_index()
    )
    metrics["top_products"] = top_products

    # Revenue by category
    by_category = (
        df.groupby("category")["total_revenue"]
        .sum()
        .reset_index()
        .sort_values("total_revenue", ascending=False)
    )
    metrics["by_category"] = by_category

    # Revenue by region
    by_region = (
        df.groupby("region")
        .agg(
            total_revenue=("total_revenue", "sum"),
            average_ticket=("total_revenue", "mean"),
            quantity=("quantity", "sum"),
        )
        .reset_index()
        .sort_values("total_revenue", ascending=False)
    )
    metrics["by_region"] = by_region
    print("\n=== AGGREGATED METRICS ===")
    for name, table in metrics.items():
        print(f"\n--- {name.upper().replace('_', ' ')} ---")
        print(table.to_string(index=False))
    
    return metrics

In [10]:
# ============================================================
# RF06 - Segment customers by spending level
# ============================================================

def segment_customers(df):
    """
    Segments customers by total spending using groupby and lambda.
    """
    customers = df.groupby("customer")["total_revenue"].sum().reset_index()
    customers.columns = ["customer", "total_spent"]
    # Lambda with conditional logic
    customers["segment"] = customers["total_spent"].apply(
        lambda spent: "Gold" if spent > 15000 else ("Silver" if spent >= 5000 else "Bronze")
    )
    customers = customers.sort_values("total_spent", ascending=False)
    print("\n=== CUSTOMER SEGMENTATION ===")
    print(customers.head(10).to_string(index=False))
    print(f"\nSegment distribution:\n{customers['segment'].value_counts()}")

    return customers

In [11]:
# ============================================================
# RF07 - Calculate statistics with NumPy
# ============================================================

def calculate_numpy_statistics(df):
    """
    Uses NumPy to calculate statistics on revenue values.
    Demonstrates:
    - conversion from Pandas column to NumPy array;
    - NumPy functions;
    - vectorized operations;
    - broadcasting.
    """
    print("\n=== NUMPY STATISTICS ===")
    revenues = df["total_revenue"].to_numpy()
    mean_revenue = np.mean(revenues)
    median_revenue = np.median(revenues)
    standard_deviation = np.std(revenues)
    total_revenue = np.sum(revenues)
    percentile_25 = np.percentile(revenues, 25)
    percentile_75 = np.percentile(revenues, 75)

    print(f"  Average revenue per sale:    ${mean_revenue:.2f}")
    print(f"  Median revenue per sale:     ${median_revenue:.2f}")
    print(f"  Standard deviation:          ${standard_deviation:.2f}")
    print(f"  Total revenue:               ${total_revenue:.2f}")
    print(f"  25th percentile (Q1):        ${percentile_25:.2f}")
    print(f"  75th percentile (Q3):        ${percentile_75:.2f}")

    # Broadcasting: Min-Max normalization
    normalized_revenues = (revenues - revenues.min()) / (revenues.max() - revenues.min())
    print(f"\n  Normalized revenues (first 5): {normalized_revenues[:5].round(4)}")

    # Vectorized operation: sales above average without loops
    above_average = revenues[revenues > mean_revenue]
    print(f"  Sales above average: {len(above_average)} out of {len(revenues)}")

    return {
        "mean": mean_revenue,
        "median": median_revenue,
        "standard_deviation": standard_deviation,
        "total": total_revenue,
        "percentile_25": percentile_25,
        "percentile_75": percentile_75,
        "sales_above_average": len(above_average),
    }


In [12]:
# ============================================================
# RF08 - Create visualizations with Matplotlib and Seaborn
# ============================================================

def generate_visualizations(df, metrics, output_dir="outputs/charts"):
    """
    Generates and exports sales visualizations as PNG files.
    """
    output_dir="outputs/charts"
    os.makedirs(output_dir, exist_ok=True)
    sns.set_theme(style="whitegrid", palette="muted")
    plt.rcParams["figure.figsize"] = (12, 6)
    plt.rcParams["axes.titlesize"] = 14
    plt.rcParams["axes.labelsize"] = 12
    # Chart 1: Total revenue by month
    fig, ax = plt.subplots()
    by_month = metrics["by_month"]
    ax.plot(by_month["month"], by_month["total_revenue"], marker="o", linewidth=2)
    ax.fill_between(by_month["month"], by_month["total_revenue"], alpha=0.15)
    ax.set_title("Total Revenue by Month")
    ax.set_xlabel("Month")
    ax.set_ylabel("Total Revenue ($)")
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(
        ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"],
        rotation=45,
    )
    plt.tight_layout()
    file_path = os.path.join(output_dir, "sales_by_month.png")
    plt.savefig(file_path, dpi=150)
    plt.close()
    print(f"  Chart exported: {file_path}")
    # Chart 2: Top 5 products by revenue
    fig, ax = plt.subplots()
    top = metrics["top_products"]
    sns.barplot(data=top, y="product", x="total_revenue", ax=ax)
    ax.set_title("Top 5 Products by Total Revenue")
    ax.set_xlabel("Total Revenue ($)")
    ax.set_ylabel("Product")
    for container in ax.containers:
        ax.bar_label(container, fmt="$ %.0f", padding=5)
    plt.tight_layout()
    file_path = os.path.join(output_dir, "top_products.png")
    plt.savefig(file_path, dpi=150)
    plt.close()
    print(f"  Chart exported: {file_path}")
    # Chart 3: Revenue distribution by region
    fig, ax = plt.subplots()
    sns.boxplot(data=df, x="region", y="total_revenue", ax=ax)
    ax.set_title("Revenue Distribution per Transaction by Region")
    ax.set_xlabel("Region")
    ax.set_ylabel("Revenue per Sale ($)")
    plt.xticks(rotation=30)
    plt.tight_layout()
    file_path = os.path.join(output_dir, "regional_distribution.png")
    plt.savefig(file_path, dpi=150)
    plt.close()
    print(f"  Chart exported: {file_path}")
    print("\n=== VISUALIZATIONS GENERATED SUCCESSFULLY ===")

In [13]:
# ============================================================
# RF09 - Create a class for the pipeline
# ============================================================

class SalesAnalyzer:
    """
    Class responsible for encapsulating the sales analysis pipeline.
    It stores the DataFrame and intermediate results as instance attributes.
    """

    def __init__(self, file_path):
        """
        Initializes the analyzer with the dataset file path.
        """
        self.file_path = file_path
        self.raw_df = None
        self.clean_df = None
        self.metrics = {}
        self.customers = None
        self.cleaning_report = {}
        self.numpy_stats = {}

    def load_data(self):
        """
        Reads the CSV file and stores the raw DataFrame.
        """
        self.raw_df = pd.read_csv(self.file_path)

        print(f"\n[SalesAnalyzer] File loaded: {self.file_path}")
        print(f"  Loaded records: {len(self.raw_df)}")

        inspect_data(self.raw_df)

        return self

    def clean_data(self):
        """
        Cleans the data and stores the processed DataFrame.
        """
        self.clean_df, self.cleaning_report = clean_data(self.raw_df.copy())

        return self

    def transform_data(self):
        """
        Applies transformations and creates derived columns.
        """
        self.clean_df = create_derived_columns(self.clean_df)
        self.clean_df = apply_extra_lambdas(self.clean_df)

        return self

    def analyze_data(self):
        """
        Calculates metrics, customer segmentation, and NumPy statistics.
        """
        self.metrics = calculate_metrics(self.clean_df)
        self.customers = segment_customers(self.clean_df)
        self.numpy_stats = calculate_numpy_statistics(self.clean_df)

        return self

    def generate_visualizations(self):
        """
        Generates and exports charts.
        """
        generate_visualizations(self.clean_df, self.metrics)

        return self

    def export_summary_report(self, file_path="outputs/summary_report.csv"):
        """
        Exports the monthly metrics report to CSV.
        """
        os.makedirs("outputs", exist_ok=True)

        self.metrics["by_month"].to_csv(
            file_path,
            index=False,
            encoding="utf-8-sig"
        )

        print(f"\n[SalesAnalyzer] Summary report exported: {file_path}")

        return self

    def summary(self):
        """
        Displays an executive summary of the pipeline.
        """
        print("\n" + "=" * 50)
        print(" SALESINSIGHT PY – EXECUTIVE SUMMARY")
        print("=" * 50)

        print(f" Dataset analyzed: {self.file_path}")

        print(
            f" Initial records: "
            f"{self.cleaning_report.get('initial_records', 'N/A')}"
        )

        print(
            f" Final records: "
            f"{self.cleaning_report.get('final_records', 'N/A')}"
        )

        total_revenue = (
            self.clean_df["total_revenue"].sum()
            if self.clean_df is not None
            else 0
        )

        print(f" Annual total revenue: ${total_revenue:,.2f}")

        if self.customers is not None and not self.customers.empty:
            top_customer = self.customers.iloc[0]

            print(
                f" Top customer: "
                f"{top_customer['customer']} "
                f"(${top_customer['total_spent']:,.2f})"
            )

        print("=" * 50)

        return self

In [14]:
# ============================================================
# RF10 - Heritage
# ============================================================
class SalesAnalyzerWithForecast(SalesAnalyzer):
    """
    Extension of SalesAnalyzer with a simple revenue forecasting feature.
    It inherits all parent class methods and adds forecasting methods.
    """

    def __init__(self, file_path, forecast_months=3):
        super().__init__(file_path)
        self.forecast_months = forecast_months
        self.forecasts = []

    def forecast_revenue_trend(self):
        """
        Forecasts revenue for the next months using the average of the last 3 months.
        This is a simple method without machine learning.
        """
        if not self.metrics or "by_month" not in self.metrics:
            print("[WARNING] Run .analyze_data() before forecasting.")
            return self

        by_month = self.metrics["by_month"].sort_values("month")
        historical_revenues = by_month["total_revenue"].to_numpy()

        last_3_months = historical_revenues[-3:]
        moving_average = np.mean(last_3_months)
        trend_factor = np.std(last_3_months) * 0.1
        last_month = int(by_month["month"].max())

        print("\n=== REVENUE TREND FORECAST ===")
        print(f"  Base: last 3 months moving average = ${moving_average:,.2f}")

        self.forecasts = []

        for i in range(1, self.forecast_months + 1):
            forecasted_month = (last_month + i - 1) % 12 + 1
            forecasted_revenue = moving_average + (trend_factor * i)

            self.forecasts.append(
                {
                    "month": forecasted_month,
                    "forecasted_revenue": round(float(forecasted_revenue), 2),
                }
            )

            print(f"  Month {forecasted_month:02d}: ${forecasted_revenue:,.2f}")

        return self

    def show_forecast_details(self):
        """
        Displays the calculated revenue forecasts.
        """
        if not self.forecasts:
            print("[WARNING] No forecast available. Run .forecast_revenue_trend() first.")
            return self

        print("\n=== FORECAST DETAILS ===")
        for forecast in self.forecasts:
            print(f"  Month {forecast['month']:02d}: ${forecast['forecasted_revenue']:,.2f}")

        return self

In [16]:
# ============================================================
# RF11 - Lambda functions and higher-order function
# ============================================================

def process_column(df, column, transformation_function):
    """
    Applies a transformation function to a DataFrame column.

    This function demonstrates the concept of a higher-order function,
    because it receives another function as an argument.
    """
    df = df.copy()
    new_column = f"{column}_transformed"
    df[new_column] = df[column].apply(transformation_function)
    print(f"  Column '{new_column}' created successfully.")
    return df


def apply_extra_lambdas(df):
    """
    Demonstrates additional lambda uses in distinct contexts.
    """
    print("\n=== LAMBDA FUNCTIONS AND CALLBACK ===")

    # Lambda in apply to create discount
    df["discount"] = df["total_revenue"].apply(lambda x: 0.10 if x > 10000 else 0.05)

    # Higher-order function using lambda as callback
    df = process_column(df, "total_revenue", lambda x: round(x / 1000, 2))
    df = process_column(df, "quantity", lambda x: "High" if x > 5 else "Low")

    # Lambda in sorted to sort a list of dictionaries
    product_list = df.groupby("product")["total_revenue"].sum().reset_index().to_dict("records")
    sorted_products = sorted(product_list, key=lambda p: p["total_revenue"], reverse=True)

    print("\nProducts sorted by revenue using sorted + lambda:")
    print(sorted_products[:3])

    return df

In [17]:
# ============================================================
# RF12 - Read and Write CSV and JSON Files
# ============================================================

def export_results(metrics, customers, numpy_stats):
    """
    Exports results as CSV and JSON files.

    Requirements covered:
    - CSV export using to_csv()
    - JSON export using json.dump()
    - JSON reading using json.load()
    """

    # Create output directory if it does not exist
    os.makedirs("outputs", exist_ok=True)

    # ========================================================
    # Export monthly metrics to CSV
    # ========================================================

    monthly_metrics_path = "outputs/metrics_by_month.csv"

    metrics["by_month"].to_csv(
        monthly_metrics_path,
        index=False,
        encoding="utf-8-sig"
    )

    print(f"  CSV exported: {monthly_metrics_path}")

    # ========================================================
    # Export customer segmentation to CSV
    # ========================================================

    customer_segmentation_path = "outputs/customer_segmentation.csv"

    customers.to_csv(
        customer_segmentation_path,
        index=False,
        encoding="utf-8-sig"
    )

    print(f"  CSV exported: {customer_segmentation_path}")

    # ========================================================
    # Export general statistics to JSON
    # ========================================================

    json_path = "outputs/general_statistics.json"

    # Convert NumPy values into JSON-serializable values
    serializable_stats = {
        key: round(float(value), 2)
        for key, value in numpy_stats.items()
    }

    with open(json_path, "w", encoding="utf-8") as file:
        json.dump(
            serializable_stats,
            file,
            indent=4,
            ensure_ascii=False
        )

    print(f"  JSON exported: {json_path}")

    # ========================================================
    # Read the exported JSON again to validate the operation
    # ========================================================

    with open(json_path, "r", encoding="utf-8") as file:
        loaded_data = json.load(file)

    print("\n  Exported JSON content:")
    print(
        json.dumps(
            loaded_data,
            indent=2,
            ensure_ascii=False
        )
    )

In [18]:
# ============================================================
# RF13 - Use regular expressions for data cleaning
# ============================================================

def clean_strings_with_regex(df):
    """
    Uses regular expressions to clean text columns.

    Examples:
    - Remove special characters
    - Standardize customer names
    - Validate customer ID format
    """

    # ========================================================
    # 1. Remove non-alphanumeric characters from customer names
    # (except underscore and spaces)
    # ========================================================

    df["clean_customer"] = df["customer"].apply(
        lambda s: re.sub(
            r"[^a-zA-Z0-9_ ]",
            "",
            str(s)
        ).strip()
    )

    # ========================================================
    # 2. Validate customer ID format
    # Expected pattern: Customer_XXX
    # Example: Customer_001, Customer_125, Customer_999
    # ========================================================

    customer_pattern = re.compile(r"^Customer_\d{3}$")

    df["valid_customer"] = df["clean_customer"].apply(
        lambda s: bool(customer_pattern.match(s))
    )

    invalid_count = (~df["valid_customer"]).sum()

    # ========================================================
    # Display cleaning results
    # ========================================================

    print("\n=== REGEX DATA CLEANING ===")
    print(
        f" Invalid customer IDs found: {invalid_count}"
    )

    print(
        f" Sample cleaned customers: "
        f"{df['clean_customer'].head(5).tolist()}"
    )

    return df

In [19]:
# ============================================================
# RF14 - Run the full pipeline
# ============================================================

def main():
    """
    Main function: runs the full SalesInsight PY pipeline.
    """
    print("\n" + "=" * 60)
    print("   SALESINSIGHT PY – Sales Data Analysis Pipeline")
    print("=" * 60)

    # Step 0: Generate dataset if necessary
    if not os.path.exists("sales.csv"):
        print("\n[INFO] Generating synthetic dataset...")
        generate_and_save_dataset(file_path="sales.csv", n_records=200)
        #generated_df = generate_sales_dataset(n_records=200)
        #generated_df.to_csv("sales.csv", index=False, encoding="utf-8-sig")

    # Steps 1 to 6: Pipeline through class with inheritance
    analyzer = SalesAnalyzerWithForecast("sales.csv", forecast_months=3)

    (
        analyzer
        .load_data()
        .clean_data()
        .transform_data()
        .analyze_data()
        .forecast_revenue_trend()
        .generate_visualizations()
        .export_summary_report()
    )

    # Extra step: regex cleaning
    analyzer.clean_df = clean_strings_with_regex(analyzer.clean_df)

    # Extra step: JSON export
    stats = calculate_numpy_statistics(analyzer.clean_df)
    export_results(analyzer.metrics, analyzer.customers, stats)

    # Final summary
    analyzer.summary()
    analyzer.show_forecast_details()

    print("\n[COMPLETED] Pipeline finished successfully!")


if __name__ == "__main__":
    main()


   SALESINSIGHT PY – Sales Data Analysis Pipeline

[INFO] Generating synthetic dataset...

=== DATASET GENERATED ===
File saved at: sales.csv
Dataset generated with 200 records.
   sale_id   sale_date      customer     product        category     region  \
0        1  2024-05-20  Customer_035       Mouse     Peripherals  Southeast   
1        2  2024-02-17  Customer_042    Keyboard     Peripherals      North   
2        3  2024-05-22  Customer_022     Monitor       Computers  Northeast   
3        4  2024-06-21  Customer_017  Smartphone  Mobile Devices  Southeast   
4        5  2024-07-12  Customer_024       Mouse     Peripherals      North   

   quantity  unit_price  
0       2.0      102.90  
1       7.0      214.88  
2       4.0         NaN  
3       4.0     2501.76  
4       8.0      121.30  

[SalesAnalyzer] File loaded: sales.csv
  Loaded records: 200

=== INITIAL DATASET INSPECTION ===
Shape: (200, 8)

Columns: ['sale_id', 'sale_date', 'customer', 'product', 'category', 'regio

C:\Users\renan\AppData\Local\Temp\ipykernel_16528\3628468436.py:20: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_columns = df.select_dtypes(include="object").columns


  Chart exported: outputs/charts\sales_by_month.png
  Chart exported: outputs/charts\top_products.png
  Chart exported: outputs/charts\regional_distribution.png

=== VISUALIZATIONS GENERATED SUCCESSFULLY ===

[SalesAnalyzer] Summary report exported: outputs/summary_report.csv

=== REGEX DATA CLEANING ===
 Invalid customer IDs found: 0
 Sample cleaned customers: ['Customer_035', 'Customer_042', 'Customer_017', 'Customer_024', 'Customer_025']

=== NUMPY STATISTICS ===
  Average revenue per sale:    $6855.18
  Median revenue per sale:     $3899.61
  Standard deviation:          $6673.94
  Total revenue:               $1233932.92
  25th percentile (Q1):        $1267.56
  75th percentile (Q3):        $10976.68

  Normalized revenues (first 5): [0.     0.042  0.3168 0.0247 0.1162]
  Sales above average: 74 out of 180
  CSV exported: outputs/metrics_by_month.csv
  CSV exported: outputs/customer_segmentation.csv
  JSON exported: outputs/general_statistics.json

  Exported JSON content:
{
  "me